In [0]:
# Silver — Chicago Cleaning
## Source  : food_inspection_group8.bronze.chicago_raw
## Target  : food_inspection_group8.silver.chicago_clean
## Rules   : Null checks, zip format, risk extract, score derivation, violation parse

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from datetime import datetime

# Detect catalog dynamically 
# Step 1 — ask Spark what the current catalog is
CATALOG = spark.sql("SELECT current_catalog()").first()[0]

# Step 2 — if it came back as a generic default, don't trust it
if CATALOG in ("hive_metastore", "spark_catalog"):

    # Step 3 — list ALL catalogs in the workspace
    catalogs = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
    # e.g. ["hive_metastore", "system", "samples", "food_inspection_group8"]

    # Step 4 — filter out the known system catalogs
    project_cats = [c for c in catalogs
                    if c not in ("hive_metastore", "spark_catalog",
                                 "system", "samples", "workspace")]
    # result → ["food_inspection_group8"]

    # Step 5 — use the first project catalog found
    CATALOG = project_cats[0] if project_cats else CATALOG
    # CATALOG = "food_inspection_group8"

# Table references
BRONZE_TABLE  = f"{CATALOG}.bronze.chicago_raw"
SILVER_TABLE  = f"{CATALOG}.silver.chicago_clean"
SILVER_VIOLS  = f"{CATALOG}.silver.chicago_violations"
QUARANTINE    = f"{CATALOG}.silver.chicago_quarantine"

# Run metadata 
RUN_TS   = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
RUN_DATE = datetime.now().strftime("%Y-%m-%d")

# Chicago zip range (IL)
CHI_ZIP_MIN = 60007
CHI_ZIP_MAX = 60827

print(f"Catalog       : {CATALOG}")
print(f"Bronze source : {BRONZE_TABLE}")
print(f"Silver target : {SILVER_TABLE}")
print(f"Run time      : {RUN_TS}") 

In [0]:
#  Verify bronze table exists 
try:
    spark.table(BRONZE_TABLE).limit(1).collect()
    print(f"✓ Bronze table found: {BRONZE_TABLE}")
except Exception as e:
    raise Exception(f"Bronze table not found: {BRONZE_TABLE}\nError: {e}")

# Get latest batch ID 
latest_batch = (
    spark.table(BRONZE_TABLE)
    .orderBy(F.col("_extract_ts").desc())
    .select("_batch_id")
    .first()[0]
)
print(f"Latest batch  : {latest_batch}")

# Select only columns needed — no Bronze audit clutter 
CHI_READ_COLS = [
    "inspection_id", "restaurant_name", "aka_name",
    "license_number", "facility_type", "risk_category",
    "street_address", "city", "state", "zip_code",
    "inspection_date", "inspection_type", "inspection_result",
    "violations_raw", "latitude", "longitude",
    "_batch_id"
]

df_chi = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("_batch_id") == latest_batch)
    .select(CHI_READ_COLS)
    # Dedup on inspection_id + violations_raw as required
    .dropDuplicates(["inspection_id", "violations_raw"])
)

row_count = df_chi.count()
print(f"Rows after dedup : {row_count:,}")
print(f"Columns          : {len(df_chi.columns)}")
display(df_chi.limit(5))

In [0]:
# Step 1: Cast all data types 
df_chi_typed = (
    df_chi
    .withColumn("inspection_date",
        F.to_date(F.col("inspection_date"), "MM/dd/yyyy"))
    .withColumn("latitude",
        F.col("latitude").cast(DoubleType()))
    .withColumn("longitude",
        F.col("longitude").cast(DoubleType()))
    .withColumn("license_number",
        F.col("license_number").cast("long").cast("string"))
)

# Step 2: Clean zip — remove .0, extract 5 digits, cast to int→str 
df_chi_typed = (
    df_chi_typed
    .withColumn("zip_code",
        F.regexp_extract(F.col("zip_code"), r"(\d{5})", 1))
    .withColumn("zip_code",
        F.col("zip_code").cast("int").cast("string"))
)

# Step 3: Extract risk → High / Medium / Low / Unknown 
df_chi_typed = df_chi_typed.withColumn(
    "risk_category",
    F.when(F.col("risk_category").contains("High"),   F.lit("High"))
     .when(F.col("risk_category").contains("Medium"), F.lit("Medium"))
     .when(F.col("risk_category").contains("Low"),    F.lit("Low"))
     .otherwise(F.lit("Unknown"))
)

# Step 4: Derive inspection_score from Results
df_chi_typed = df_chi_typed.withColumn(
    "inspection_score",
    F.when(F.col("inspection_result") == "Pass",               F.lit(90.0))
     .when(F.col("inspection_result") == "Pass w/ Conditions", F.lit(80.0))
     .when(F.col("inspection_result") == "Fail",               F.lit(70.0))
     .when(F.col("inspection_result") == "No Entry",           F.lit(0.0))
     .otherwise(F.lit(None).cast(DoubleType()))
)

# Step 5: Add city_source 
df_chi_typed = df_chi_typed.withColumn("city_source", F.lit("CHI"))

# Step 6: Early filter — every inspection must have at least 1 violation
# Assignment rule: no exceptions, including Pass results
before = df_chi_typed.count()

df_chi_typed = df_chi_typed.filter(
    F.col("violations_raw").isNotNull() &
    (F.col("violations_raw") != "")
)

after  = df_chi_typed.count()
dropped = before - after

print(f"Before violation filter : {before:,}")
print(f"After  violation filter : {after:,}")
print(f"Dropped (no violations) : {dropped:,}")
print(f"  → These are inspections with null/empty violations_raw")
print(f"  → Includes Pass inspections with no recorded violations")

display(df_chi_typed.select(
    "inspection_id", "restaurant_name", "risk_category",
    "inspection_result", "inspection_score",
    "zip_code", "inspection_date"
).limit(5))

In [0]:
# Apply all DQX checks as named boolean columns 
df_chi_checked = (
    df_chi_typed

    # Check 1: valid_business_name
    .withColumn("valid_business_name",
        F.col("restaurant_name").isNotNull() &
        (F.col("restaurant_name") != ""))

    # Check 2: valid_inspection_date
    .withColumn("valid_inspection_date",
        F.col("inspection_date").isNotNull())

    # Check 3: valid_inspection_type
    .withColumn("valid_inspection_type",
        F.col("inspection_type").isNotNull() &
        (F.col("inspection_type") != ""))

    # Check 4: valid_zip_code (not null)
    .withColumn("valid_zip_code",
        F.col("zip_code").isNotNull() &
        (F.col("zip_code") != "") &
        (F.col("zip_code") != "null"))

    # Check 5: valid_zip_format — exactly as specified ^[0-9]+$
    .withColumn("valid_zip_format",
        F.col("zip_code").rlike(r"^[0-9]+$"))

    # Check 6: valid_zip_range — Chicago IL zips 60007–60827
    .withColumn("valid_zip_range",
        F.col("zip_code").cast("int").between(
            CHI_ZIP_MIN, CHI_ZIP_MAX))

    # Check 7: valid_results — Chicago results cannot be null
    .withColumn("valid_results",
        F.col("inspection_result").isNotNull() &
        (F.col("inspection_result") != ""))

    # Check 8: no urgent/critical on passing inspections
    # null violations = auto True (no violations = no urgent terms)
    .withColumn("valid_no_urgent_pass",
        F.when(F.col("violations_raw").isNull(), F.lit(True))
         .otherwise(
             ~(F.col("inspection_result").isin(
                   "Pass", "Pass w/ Conditions") &
               F.col("violations_raw").rlike(r"(?i)(urgent|critical)"))
         ))

    # Overall: ALL checks must pass
    .withColumn("_all_checks_pass",
        F.col("valid_business_name")    &
        F.col("valid_inspection_date")  &
        F.col("valid_inspection_type")  &
        F.col("valid_zip_code")         &
        F.col("valid_zip_format")       &
        F.col("valid_zip_range")        &
        F.col("valid_results")          &
        F.col("valid_no_urgent_pass")
    )
)

# Check column list
CHECK_COLS = [
    "valid_business_name", "valid_inspection_date",
    "valid_inspection_type", "valid_zip_code",
    "valid_zip_format", "valid_zip_range",
    "valid_results", "valid_no_urgent_pass",
    "_all_checks_pass"
]

# Split good vs bad
df_chi_good = (
    df_chi_checked
    .filter(F.col("_all_checks_pass") == True)
    .drop(*CHECK_COLS)
    .withColumn("_silver_ts",   F.lit(RUN_TS))
    .withColumn("_silver_date", F.lit(RUN_DATE))
    .withColumn("_dqx_status",  F.lit("PASS"))
)

df_chi_bad = (
    df_chi_checked
    .filter(
        F.col("_all_checks_pass").isNull() |
        (F.col("_all_checks_pass") == False)
    )
    .withColumn("_silver_ts",   F.lit(RUN_TS))
    .withColumn("_silver_date", F.lit(RUN_DATE))
    .withColumn("_dqx_status",  F.lit("FAIL"))
)

# Verify zero rows lost
original   = df_chi_typed.count()
good_count = df_chi_good.count()
bad_count  = df_chi_bad.count()
lost       = original - (good_count + bad_count)

print(f"Original rows : {original:,}")
print(f"GOOD rows     : {good_count:,}")
print(f"BAD rows      : {bad_count:,}")
print(f"Rows lost     : {lost}  ← must be 0")

# Show which checks are failing
print("\nFailed check breakdown:")
display(
    df_chi_bad.select(*CHECK_COLS)
    .groupBy(*[c for c in CHECK_COLS if c != "_all_checks_pass"])
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
)

In [0]:
# Parse pipe-delimited violations → long format 
df_chi_viols = (
    df_chi_good
    .filter(F.col("violations_raw").isNotNull())
    .withColumn("viol_entry",
        F.explode(F.split(F.col("violations_raw"), r"\|")))
    .withColumn("viol_entry",
        F.trim(F.col("viol_entry")))
    .filter(F.col("viol_entry") != "")
    # Extract violation code — number before first period
    .withColumn("violation_code",
        F.regexp_extract(F.col("viol_entry"), r"^(\d+)\.", 1))
    # Extract description — between code and "- Comments:"
    .withColumn("violation_description",
        F.trim(F.regexp_extract(F.col("viol_entry"),
            r"^\d+\.\s*(.+?)(?:\s*-\s*Comments:.*)?$", 1)))
    # Extract inspector comment
    .withColumn("inspector_comment",
        F.regexp_extract(F.col("viol_entry"),
            r"-\s*Comments:\s*(.+)", 1))
    # Drop blank codes
    .filter(F.col("violation_code") != "")
    # Dedup: one row per inspection + violation code
    .dropDuplicates(["inspection_id", "violation_code"])
    # Keep only violation-relevant columns
    .select(
        "inspection_id", "restaurant_name",
        "inspection_date", "city_source",
        "violation_code", "violation_description",
        "inspector_comment", "_batch_id"
    )
)

#  Count distinct violations per inspection 
viol_counts = (
    df_chi_viols
    .groupBy("inspection_id")
    .agg(F.countDistinct("violation_code").alias("violation_count"))
)

#Join count back — alias prevents column collision 
df_chi_good = (
    df_chi_good.alias("g")
    .join(viol_counts.alias("v"), on="inspection_id", how="left")
    .withColumn("violation_count",
        F.coalesce(F.col("v.violation_count"), F.lit(0)))
    .select(
        *[F.col(f"g.{c}") for c in df_chi_good.columns],
        F.col("violation_count")
    )
)

print(f"Good inspections (with counts) : {df_chi_good.count():,}")
print(f"Violation rows parsed          : {df_chi_viols.count():,}")

display(df_chi_viols.select(
    "inspection_id", "violation_code",
    "violation_description", "inspector_comment"
).limit(5))

In [0]:
#  Write 1: Clean inspections 
(
    df_chi_good
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)
print(f"✓ {SILVER_TABLE}")
print(f"  Rows    : {df_chi_good.count():,}")
print(f"  Columns : {len(df_chi_good.columns)}")

#  Write 2: Violations long format 
(
    df_chi_viols
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_VIOLS)
)
print(f"\n✓ {SILVER_VIOLS}")
print(f"  Rows    : {df_chi_viols.count():,}")

# Write 3: Quarantine 
(
    df_chi_bad
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE)
)
print(f"\n✓ {QUARANTINE}")
print(f"  Rows    : {df_chi_bad.count():,}")
print(f"\n_silver_ts  : {RUN_TS}")
print(f"_silver_date: {RUN_DATE}")

In [0]:
# Dynamic verify — no hardcoded catalog name
df_verify = spark.sql(f"""
    SELECT
        city_source,
        COUNT(*)                                            AS total_inspections,
        ROUND(AVG(inspection_score), 2)                     AS avg_score,
        ROUND(AVG(violation_count), 2)                      AS avg_violations,
        SUM(CASE WHEN inspection_result = 'Pass'
                 THEN 1 ELSE 0 END)                         AS pass_count,
        SUM(CASE WHEN inspection_result = 'Fail'
                 THEN 1 ELSE 0 END)                         AS fail_count,
        SUM(CASE WHEN inspection_result = 'Pass w/ Conditions'
                 THEN 1 ELSE 0 END)                         AS pass_cond_count,
        SUM(CASE WHEN risk_category = 'High'
                 THEN 1 ELSE 0 END)                         AS high_risk,
        SUM(CASE WHEN risk_category = 'Medium'
                 THEN 1 ELSE 0 END)                         AS medium_risk,
        SUM(CASE WHEN risk_category = 'Low'
                 THEN 1 ELSE 0 END)                         AS low_risk,
        SUM(CASE WHEN risk_category = 'Unknown'
                 THEN 1 ELSE 0 END)                         AS unknown_risk,
        MIN(_silver_ts)                                     AS first_loaded,
        MAX(_silver_ts)                                     AS last_loaded
    FROM {SILVER_TABLE}
    GROUP BY city_source
""")
display(df_verify)

In [0]:
# Show which checks caused failures — proof of DQX-style validation
df_quar = spark.sql(f"""
    SELECT
        SUM(CASE WHEN valid_business_name   = false THEN 1 ELSE 0 END) AS failed_business_name,
        SUM(CASE WHEN valid_inspection_date = false THEN 1 ELSE 0 END) AS failed_date,
        SUM(CASE WHEN valid_inspection_type = false THEN 1 ELSE 0 END) AS failed_type,
        SUM(CASE WHEN valid_zip_code        = false THEN 1 ELSE 0 END) AS failed_zip_null,
        SUM(CASE WHEN valid_zip_format      = false THEN 1 ELSE 0 END) AS failed_zip_format,
        SUM(CASE WHEN valid_zip_range       = false THEN 1 ELSE 0 END) AS failed_zip_range,
        SUM(CASE WHEN valid_results         = false THEN 1 ELSE 0 END) AS failed_results,
        SUM(CASE WHEN valid_no_urgent_pass  = false THEN 1 ELSE 0 END) AS failed_urgent_pass,
        COUNT(*)                                                         AS total_quarantined
    FROM {QUARANTINE}
""")
display(df_quar)

# Delta history — shows timestamp of every write
display(spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}"))